# ASL v1 — Encoder Pretraining + Fine-tune (Colab T4)

Offloads the **500-class encoder pretraining** and the **75-class fine-tune** to a
free Colab T4 GPU, so the local machine stays free for other work.

### One-time setup
1. `Runtime → Change runtime type → T4 GPU`.
2. Locally run `python -m asl.package_for_colab` and upload **three files** to
   a Drive folder `MyDrive/asl-model/`:
   - `code_bundle.zip`  (the `asl` package + configs + manifest/norm/splits)
   - `pretrain_jpeg.npz`  (~0.9 GB — the 500-class pretrain frames, JPEG-packed)
   - `clips.npz`  (~1.1 GB — the 75-class fine-tune frames)

All real logic lives in the `asl` package (identical to local); this notebook
only orchestrates it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, zipfile, shutil
DRIVE = '/content/drive/MyDrive/asl-model'
os.makedirs('/content/work/artifacts/cache', exist_ok=True)
with zipfile.ZipFile(f'{DRIVE}/code_bundle.zip') as z:
    z.extractall('/content/work')
# copy the two data caches onto fast local disk (Drive I/O is slow)
for f in ['pretrain_jpeg.npz', 'clips.npz']:
    shutil.copy(f'{DRIVE}/{f}', f'/content/work/artifacts/cache/{f}')
%cd /content/work
!pip -q install pyyaml onnx onnxruntime
import torch; print('cuda available:', torch.cuda.is_available())

In [ ]:
# Decode the JPEG-packed pretrain cache back to frames.dat (identical to local)
!PYTHONPATH=src python -m asl.pack_pretrain_jpeg --unpack \
    --in artifacts/cache/pretrain_jpeg.npz --cache artifacts/cache/pretrain

In [ ]:
# Pretrain the encoder from scratch on the 500-class set (CUDA auto-selected).
!PYTHONPATH=src python -u -m asl.pretrain --cache artifacts/cache/pretrain \
    --epochs 45 --warmup 4 --batch-size 64 --lr 0.004 \
    --out artifacts/checkpoints/pretrain
import shutil, os
os.makedirs('/content/drive/MyDrive/asl-model/out', exist_ok=True)
shutil.copy('artifacts/checkpoints/pretrain/encoder.pt',
            '/content/drive/MyDrive/asl-model/out/encoder.pt')
print('saved encoder.pt to Drive')

In [ ]:
# Fine-tune the 75-class head on the pretrained encoder, then save results.
!PYTHONPATH=src python -u -m asl.train --config configs/finetune.yaml
import shutil, os
OUT = '/content/drive/MyDrive/asl-model/out'
for f in ['artifacts/checkpoints/finetune/best.pt',
          'artifacts/checkpoints/finetune/history.json']:
    shutil.copy(f, f'{OUT}/{os.path.basename(f)}')
print('fine-tune done; best.pt + history.json copied to Drive')